# 02 — Additional continual-training recovery checks

This bounded companion tests historical actor cache restoration, cumulative stage continuation, and malformed manifests using disposable real-search runs. It uses the same frozen trainer definitions and actual GNU opponent calls. It never modifies scientific training runs.

**Inputs:** notebooks 00–02 and pinned prior B training trajectories. **Outputs:** disposable `checkpoints/ahr-v4-extra-recovery-*` and `reports/ahr_native_v4/training_recovery_extensions.json`. **Runtime:** default imports only; explicit validation approximately 2–5 minutes on CUDA 1. **Resume:** the validation starts uniquely named fixtures; each worker can resume with its saved configuration using notebook 02. Production runs use the regular controller. No training or evaluation strength claim is made.

The cumulative-boundary comparison ignores only the explicit boundary markers and their event counters. It requires exactly identical learned weights, optimizer/scheduler/scaler, all used RNG streams, sampler, opponents, metrics and game trajectories.

In [ ]:
from pathlib import Path
import os, nbformat
PROJECT=Path(os.environ.get('AHRV4_WORKSPACE',Path.cwd())).resolve()
while not (PROJECT/'notebooks/00_instance_setup_and_recovery.ipynb').exists():
    if PROJECT.parent==PROJECT:raise RuntimeError('Run within the AHR Go workspace')
    PROJECT=PROJECT.parent
_shared=nbformat.read(PROJECT/'experiments/ahr_native_v4/00_shared.ipynb',4)
for _cell in _shared.cells:
    if _cell.cell_type=='code' and 'ahr-export' in _cell.metadata.get('tags',[]):exec(compile(_cell.source,'00_shared:ahr-export','exec'),globals())
v4_load_upstream(PROJECT,globals())
v4_import_exports(PROJECT/'experiments/ahr_native_v4/01_native_ahr_and_tactics.ipynb',globals())
v4_import_exports(PROJECT/'experiments/ahr_native_v4/02_competitive_training.ipynb',globals())

In [ ]:
def v4_validate_training_fallback(project):
    config=V4TrainConfig(scheduled_games=1,warmup_updates=0,updates_per_game=1,batch_size=4,device='cpu',
        max_plies=1,search={'time_budget_s':0.0},reanalysis_per_game=0,prior_runs=(),purpose='synthetic_smoke')
    components,provenance=v4_training_components(project,config)
    state=v4_initial_training_state(project,'synthetic-zero-budget-fallback',config,{},[],components,provenance)
    runtime={}
    for _ in range(5):v4_training_event(project,'synthetic-zero-budget-fallback',state,components,config,{'train':[]},runtime)
    game=state['current_game'];record=game['records'][0]
    if state['phase']!='ingest' or game['status']!='UNKNOWN' or game['reason']!='max_plies_cutoff':raise AssertionError('Fallback cutoff mislabeled')
    if record['root_value'] is not None or record['root_utility'] is not None:raise AssertionError('Unknown fallback fabricated a value')
    if record['policy_weight']!=0.0:raise AssertionError('Unanalyzed fallback became a policy teacher')
    if any(key in record for key in ('outcome_target','score_target','ownership_target')):raise AssertionError('Fallback supplied terminal labels')
    final=v4_verify_training_game(game)
    if final.terminal:raise AssertionError('Fixture unexpectedly terminal')
    rejected=False
    try:v4_training_example(game,0,{'origin':'current_training'})
    except ValueError:rejected=True
    if not rejected:raise AssertionError('UNKNOWN accepted as a terminal training example')
    return {'status':'PASS','zero_budget_legal_move':True,'unknown_cutoff_preserved':True,
            'root_neural_value':None,'root_utility':None,'global_policy_weight':0.0,'terminal_targets_rejected':True}

def v4_validate_recovery_extensions(project,device='cuda:1'):
    import shutil
    project=Path(project);sources=v4_source_identity(project);fallback=v4_validate_training_fallback(project);prefix='ahr-v4-extra-recovery-'+uuid.uuid4().hex[:8]
    search={'time_budget_s':30.,'max_depth':1,'max_nodes':8,'branch_width':2,'tactical_width':2,'quiescence_extensions':0}
    config=V4TrainConfig(seed=2026100213,scheduled_games=2,warmup_updates=1,updates_per_game=2,batch_size=8,
        device=device,max_plies=4,search=search,reanalysis_search=search,reanalysis_per_game=0,snapshot_every_games=1,
        purpose='synthetic_smoke',validation_scripted=False,validation_initial_game=2,prior_runs=('ahr-v3-B-s01',))
    full=prefix+'-full';mid=prefix+'-historical';staged=prefix+'-staged'
    v4_fresh_training_call(project,full,config)
    v4_fresh_training_call(project,mid,config,7)
    saved,_=v4_read_training_checkpoint(project/'checkpoints'/mid)
    if saved['phase']!='commit_move' or saved['pending_move']['actor']!='historical':raise AssertionError('Historical cached actor boundary not exercised')
    if len(saved['current_game']['records'])!=1:raise AssertionError('Unexpected pending historical move position')
    v4_fresh_training_call(project,mid,config)
    original,_=v4_read_training_checkpoint(project/'checkpoints'/full);restored,_=v4_read_training_checkpoint(project/'checkpoints'/mid)
    a=v4_semantic_training_state(project,original);b=v4_semantic_training_state(project,restored)
    if not nested_equal(a,b):raise AssertionError('Historical actor recreation changed exact state: '+str([k for k in a if not nested_equal(a[k],b[k])]))
    v4_fresh_training_call(project,staged,config,run_until_games=1)
    boundary,_=v4_read_training_checkpoint(project/'checkpoints'/staged)
    if boundary['games_finished']!=1 or boundary['phase']!='new_game' or boundary['optimizer_step']!=3:raise AssertionError('Cumulative boundary stopped before updates completed')
    v4_fresh_training_call(project,staged,config,run_until_games=2)
    continued,_=v4_read_training_checkpoint(project/'checkpoints'/staged)
    c=v4_semantic_training_state(project,continued)
    for key in ('global_step','stages_reached'):a.pop(key);c.pop(key)
    if not nested_equal(a,c):raise AssertionError('Cumulative stage continuation changed learned state: '+str([k for k in a if not nested_equal(a[k],c[k])]))
    run_dir=project/'checkpoints'/full;manifest=json.loads((run_dir/'manifest.json').read_text());rejected={}
    malformed=project/'checkpoints'/(prefix+'-malformed');malformed.mkdir(parents=True)
    bad=copy.deepcopy(manifest);bad['latest']='missing-reference.pt';atomic_json(malformed/'manifest.json',bad)
    try:v4_read_training_checkpoint(malformed)
    except RuntimeError as error:rejected['malformed_latest']='cursor mismatch' in str(error)
    for entry in manifest['entries']:shutil.copy2(run_dir/entry['file'],malformed/entry['file'])
    bad=copy.deepcopy(manifest);bad['entries'][0]['sha256']='0'*64;atomic_json(malformed/'manifest.json',bad)
    try:v4_read_training_checkpoint(malformed)
    except RuntimeError as error:rejected['payload_hash_mismatch']='Corrupt' in str(error)
    (malformed/'manifest.json').unlink()
    try:v4_read_training_checkpoint(malformed)
    except RuntimeError as error:rejected['orphan_payload']='without manifest' in str(error)
    if len(rejected)!=3 or not all(rejected.values()):raise AssertionError('Malformed manifest rejection failed')
    if sum(original['sampler']['consumed'].values())!=original['optimizer_step']*config.batch_size:raise AssertionError('Consumed-example accounting drift')
    if original['metrics']['opponent_games']!={'historical':1,'gnu':1}:raise AssertionError('Fixture opponent mix incorrect')
    if v4_source_identity(project)!=sources:raise RuntimeError('Sources changed during extra recovery checks')
    receipt={'status':'PASS','sources':sources,'source_identity':sources,'device':device,
        'historical_actor_cache_exact_resume':True,'cumulative_stage_exact_resume':True,'nullable_fallback':fallback,
        'cumulative_ignored_fields':['explicit boundary event count','stages_reached'],
        'gnu_moves':original['metrics']['gnu_moves'],'optimizer_steps':original['optimizer_step'],
        'consumed_examples_exact':True,'manifest_rejections':rejected,
        'runs':{'full':full,'historical_resume':mid,'staged':staged},
        'validation_notebook_code_sha256':v4_code_sha(project/'experiments/ahr_native_v4/02_recovery_extensions.ipynb',True)}
    atomic_json(v4_report_path(project,'training_recovery_extensions.json'),receipt);return receipt


In [ ]:
if os.environ.get('AHRV4_EXTRA_RECOVERY','0')=='1':
    result=v4_validate_recovery_extensions(PROJECT,device='cuda:1')
    print(json.dumps({k:result[k] for k in ('status','historical_actor_cache_exact_resume','cumulative_stage_exact_resume','manifest_rejections','gnu_moves','optimizer_steps')},indent=2))
elif os.environ.get('AHRV4_EXTRA_RECOVERY')=='fallback':
    print(json.dumps(v4_validate_training_fallback(PROJECT),indent=2))
else:
    print('Ready. Set AHRV4_EXTRA_RECOVERY=1 for disposable fresh-process recovery checks.')